# Example: dispatch-derived LCOE and TEA

This standalone example loads PNM data, declares the hybrid system and its economics, dispatches it, and passes operation metrics to `LCOECalculator`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd()
while not (root / 'enliten').is_dir():
    if root.parent == root: raise RuntimeError('Run from inside the ENLITEN repository.')
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from enliten import ChargingPath, Generation, LCOECalculator, Site, Storage, System

def profile(filename):
    frame = pd.read_csv(root / 'examples' / 'data' / filename)
    return pd.Series(frame['PNM'].to_numpy(float), index=pd.to_datetime(frame.iloc[:, 0], utc=True))

demand, pv, csp = profile('PNM_demand.csv'), profile('PNM_pv_ac_1MW_av.csv'), profile('PNM_csp_th_av.csv')
hours, start = 24 * 14, pd.Timestamp('2023-07-01', tz='UTC')
i = demand.index.get_loc(start)
load = (demand.iloc[i:i + hours] * 0.05).rename('load_MW')
pv_multiplier, bes_capacity, bes_power, tes_power = 3_000.0, 750.0, 150.0, 150.0
pv_capex = pv.max() * pv_multiplier * 1_000 * 1_430
bes_capex = bes_capacity * 1_000 * 300
csp_tes_capex = tes_power * 1_000 * 7_912
site = Site('microgrid')
assets = [
    Storage('tes', site, 2_500.0, tes_power, 'thermal', 'electric', 0.50, maximum_stored_energy_rate_MW=300.0, variable_opex_USD_per_MWh=3.8),
    Generation('csp', site, csp.iloc[i:i + hours], 'thermal', False, False, capex=csp_tes_capex, opex=tes_power * 1_000 * 74.6),
    Storage('bes', site, bes_capacity, bes_power, 'electric', 'electric', 0.90, maximum_stored_energy_rate_MW=bes_power, capex=bes_capex, opex=0.025 * bes_capex),
    Generation('pv', site, pv.iloc[i:i + hours] * pv_multiplier, 'electric', capex=pv_capex, opex=pv.max() * pv_multiplier * 1_000 * 24),
]
paths = [ChargingPath('csp', 'tes', 'thermal', 'thermal', 0.90, 300.0 / 0.90), ChargingPath('pv', 'bes', 'electric', 'electric', 0.90, bes_power / 0.90)]
system = System(load, assets, paths)
system.operation_metrics()

In [ ]:
pd.DataFrame([{'asset': asset.name, 'capex_USD': asset.capex, 'fixed_OM_USD_per_year': asset.opex, 'variable_OM_USD_per_MWh': asset.variable_opex_USD_per_MWh} for asset in system.systems])

In [ ]:
lcoe_metrics = LCOECalculator.from_system(system, analysis_period=30).calculate_lcoe_metrics()
pd.Series(lcoe_metrics)